In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
WEIGHTS_DIR = "gravnet_nodes_faser_all_events_4class"  # folder under get_weights_path()
RUN         = 10000
GPU         = "cuda:0"
NUM_EVENTS  = None      # None → load all events per chunk

CLASS_NAMES = ["other", "secondary_e", "primary_EM_e", "mu"]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import seaborn as sns
import torch
from pathlib import Path
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch_geometric.loader import DataLoader
from tqdm.notebook import tqdm

from analysis.gravnet.model import NeutrinoGravNetNodesFaser
from analysis.gravnet.create_data_particle_prob_4class import evaluate, get_str_from_run
from analysis.utils.utils import get_torch_path, get_weights_path, get_figures_path
from analysis.utils.validation_utils import get_accuracy_dict

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 350,
})

COLORS = ["#4477AA", "#EE6677", "#228833", "#CCBB44"]   # one per class

device = torch.device(GPU if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

weights_path = get_weights_path() / WEIGHTS_DIR / "best_model.pt"
torch_path   = get_torch_path()
figures_path = get_figures_path() / "gravnet_nodes_4class"
figures_path.mkdir(parents=True, exist_ok=True)

print(f"Weights : {weights_path}")
print(f"Figures : {figures_path}")

In [ ]:
import seaborn as sns

sns.set_style("ticks")
sns.set_context("paper", font_scale=1.2)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif', 'Times New Roman', 'Times'],
    'mathtext.fontset': 'dejavuserif',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'figure.dpi': 350,
})

C1 = "#353D4C"   # train
C2 = "#E17883"   # val
C3 = "#5691D9"   # weighted acc
MARKER_KW = dict(markersize=4, markerfacecolor='white', markeredgewidth=1.2)

metrics_path = weights_path.parent / "training_metrics.npz"

if not metrics_path.exists():
    print(f"training_metrics.npz not found — training still in progress.")
    print(f"Expected at: {metrics_path}")
else:
    metrics = np.load(metrics_path)
    epochs  = np.arange(1, len(metrics["train_loss"]) + 1)
    best_ep = int(np.argmin(metrics["val_loss"])) + 1

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ax.plot(epochs, metrics["train_loss"], marker='o', linewidth=0.8, color=C1, label="Train", **MARKER_KW)
    ax.plot(epochs, metrics["val_loss"],   marker='s', linewidth=0.8, color=C2, label="Val",   **MARKER_KW)
    ax.axvline(best_ep, color=C2, linestyle=':', linewidth=1.0, alpha=0.7, label=f"Best (ep {best_ep})")
    ax.set_yscale("log")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy loss")
    ax.legend(frameon=False)
    ax.set_title("Loss")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(epochs, metrics["train_acc"], marker='o', linewidth=0.8, color=C1, label="Train", **MARKER_KW)
    ax.plot(epochs, metrics["val_acc"],   marker='s', linewidth=0.8, color=C2, label="Val",   **MARKER_KW)
    ax.axvline(best_ep, color=C2, linestyle=':', linewidth=1.0, alpha=0.7)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.legend(frameon=False)
    ax.set_title("Overall accuracy (unweighted)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, alpha=0.3)

    ax = axes[2]
    ax.plot(epochs, metrics["val_wacc"], marker='s', linewidth=0.8, color=C3, label="Val wacc", **MARKER_KW)
    ax.axhline(0.25, color='gray', linestyle='--', linewidth=0.8, label="Random (0.25)")
    ax.axvline(best_ep, color=C2, linestyle=':', linewidth=1.0, alpha=0.7, label=f"Best (ep {best_ep})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Weighted accuracy")
    ax.legend(frameon=False)
    ax.set_title("Weighted accuracy (val)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(figures_path / "training_curves.png", dpi=350, bbox_inches='tight')
    plt.show()
    print(f"Best epoch: {best_ep}  val_loss={metrics['val_loss'][best_ep-1]:.4f}  "
          f"val_acc={metrics['val_acc'][best_ep-1]:.4f}  val_wacc={metrics['val_wacc'][best_ep-1]:.4f}")
    print("Saved: training_curves.png")


__Load data (held-out validation set)__

In [ ]:
run_str  = get_str_from_run(RUN)
run_path = torch_path / f"{RUN}/pointnetpp_faser_all_events"

chunk_files = sorted(run_path.glob(f"{run_str}_*.pt"))
chunk_files = [f for f in chunk_files if "_particle_prob" not in f.stem]
print(f"Run {RUN} ({run_str}): {len(chunk_files)} base chunks found")

dataset = []
for chunk_file in chunk_files:
    chunk_data = torch.load(chunk_file, weights_only=False)
    if NUM_EVENTS is not None:
        chunk_data = chunk_data[:NUM_EVENTS]
    dataset.extend(chunk_data)
    print(f"  {chunk_file.name}: {len(chunk_data)} events")

print(f"\nTotal events loaded: {len(dataset)}")

# Reproducible val split — must match training (random_state=42, test_size=0.2)
_, val_dataset = train_test_split(dataset, test_size=0.2, random_state=42)
print(f"Val set            : {len(val_dataset)} events")

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

__Load model and run inference__

In [ ]:
model = NeutrinoGravNetNodesFaser(input_dim=1, num_node_classes=4, faser_dim=5).to(device)

ckpt = torch.load(weights_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Checkpoint epoch : {ckpt['epoch'] + 1}")
print(f"Val loss         : {ckpt.get('val_loss', float('nan')):.4f}")
print(f"Parameters       : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Inference cache — keyed by epoch + val_loss so re-runs on new checkpoint
_ep  = ckpt['epoch']
_vl  = ckpt.get('val_loss', 0.0)
cache_path = figures_path / f"infer_cache_ep{_ep}_vl{_vl:.4f}.npz"

if cache_path.exists():
    print(f"\nLoading inference from cache: {cache_path.name}")
    _c     = np.load(cache_path)
    y_true = _c["y_true"]
    y_pred = _c["y_pred"]
    y_prob = _c["y_prob"]
    print(f"Loaded {len(y_true):,} nodes.")
else:
    print("\nRunning inference...")
    y_true_list, y_pred_list, y_prob_list = evaluate(model, val_loader, device, use_faser=True)
    y_true = np.concatenate(y_true_list)
    y_pred = np.concatenate(y_pred_list)
    y_prob = np.concatenate(y_prob_list)
    np.savez_compressed(cache_path, y_true=y_true, y_pred=y_pred, y_prob=y_prob)
    print(f"Saved inference cache: {cache_path.name}")

print(f"\nNodes evaluated : {len(y_true):,}")
print(f"Overall accuracy: {(y_true == y_pred).mean():.4f}")


__Per-class accuracy__

In [ ]:
acc_dict = get_accuracy_dict(y_true=y_true, y_pred=y_pred, class_names=CLASS_NAMES)

print(f"{'Class':<20}  {'Accuracy':>8}")
print("-" * 32)
for name, acc in acc_dict.items():
    marker = "  ←" if name == "average" else ""
    print(f"{name:<20}  {acc * 100:>7.2f}%{marker}")

__Confusion matrix (normalised by true class)__

In [ ]:
cm      = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

# Custom annotations: normalised value + raw count
annot = np.empty_like(cm, dtype=object)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot[i, j] = f"{cm_norm[i, j]:.2f}\n({cm[i, j]:,})"

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(
    cm_norm, annot=annot, fmt="", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    vmin=0, vmax=1,
    cbar_kws={"label": "Recall (fraction of true class)"},
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Node classification — confusion matrix")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(figures_path / "confusion_matrix.png", bbox_inches="tight")
plt.show()

__Softmax probability distributions per true class__

For each true class, histogram of $P(\text{class})$ split by correct vs incorrect predictions. Helps assess calibration and decision confidence.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()
bins = np.linspace(0, 1, 31)

for true_idx, (ax, true_name) in enumerate(zip(axes, CLASS_NAMES)):
    mask         = y_true == true_idx
    y_prob_cls   = y_prob[mask]
    correct_mask = y_pred[mask] == true_idx
    prob_col     = y_prob_cls[:, true_idx]

    ax.hist(
        prob_col[correct_mask],  bins=bins, alpha=0.65,
        color=COLORS[true_idx], label=f"Correct  ({correct_mask.sum():,})",
        density=True,
    )
    ax.hist(
        prob_col[~correct_mask], bins=bins, alpha=0.50,
        color="#999999", label=f"Incorrect ({(~correct_mask).sum():,})",
        density=True,
    )
    ax.set_xlabel(f"$P(\\mathtt{{{true_name}}})$", fontsize=10)
    ax.set_ylabel("Density")
    ax.set_title(f"True class: {true_name}")
    ax.legend(fontsize=8)

plt.suptitle("Predicted softmax probability for each true class", y=1.01)
plt.tight_layout()
plt.savefig(figures_path / "probability_distributions.png", bbox_inches="tight")
plt.show()

__Per-class ROC curves (one-vs-rest)__

In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(4)))  # (N, 4)

fig, ax = plt.subplots(figsize=(6, 5))
for i, (name, color) in enumerate(zip(CLASS_NAMES, COLORS)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=1.8, label=f"{name}  (AUC = {roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("One-vs-rest ROC curves")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(figures_path / "roc_curves.png", bbox_inches="tight")
plt.show()

__Example events — node-level predictions__

Scatter plot: $x$ vs $z$ position, nodes coloured by **predicted** class. Misclassified nodes are marked with ✕.

In [ ]:
N_EXAMPLES = 4
sample_indices = np.linspace(0, len(val_dataset) - 1, N_EXAMPLES, dtype=int)

fig, axes = plt.subplots(1, N_EXAMPLES, figsize=(4.5 * N_EXAMPLES, 4))
model.eval()
with torch.no_grad():
    for ax_idx, ev_idx in enumerate(sample_indices):
        data    = val_dataset[int(ev_idx)]
        data_d  = data.clone().to(device)
        out     = model(data_d.x, data_d.pos, data_d.batch, data_d.x_faser)
        pred    = out.argmax(dim=1).cpu().numpy()
        true    = data.pdg_label.numpy()
        pos     = data.pos.numpy()          # (N, 3): columns are [x, y, z] or similar
        correct = pred == true

        ax = axes[ax_idx]
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            cls_mask = pred == cls_idx
            # Correct predictions
            sel = cls_mask & correct
            if sel.sum() > 0:
                ax.scatter(
                    pos[sel, 2], pos[sel, 0],
                    c=COLORS[cls_idx], s=6, alpha=0.8,
                    label=cls_name, zorder=2, linewidths=0,
                )
            # Misclassified — X marker
            sel = cls_mask & ~correct
            if sel.sum() > 0:
                ax.scatter(
                    pos[sel, 2], pos[sel, 0],
                    c=COLORS[cls_idx], s=35, marker="x", alpha=1.0,
                    zorder=3, linewidths=0.8,
                )

        ax.set_xlabel("$z$ (mm)")
        ax.set_ylabel("$x$ (mm)")
        ax.set_title(f"Val event {ev_idx}\nacc = {correct.mean():.2f}")

# Shared legend
handles = [
    mlines.Line2D([], [], marker="o", color="w", markerfacecolor=COLORS[i],
                  markersize=7, label=n)
    for i, n in enumerate(CLASS_NAMES)
]
handles.append(
    mlines.Line2D([], [], marker="x", color="#444444", markersize=7,
                  linestyle="None", label="Misclassified")
)
fig.legend(handles=handles, loc="upper right",
           bbox_to_anchor=(1.0, 1.02), fontsize=9)

plt.tight_layout()
plt.savefig(figures_path / "example_events.png", bbox_inches="tight")
plt.show()

__Failure analysis — off-diagonal confusion breakdown__

For each true class, what fraction of its misclassified nodes are predicted as each other class.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for true_idx, (ax, true_name) in enumerate(zip(axes, CLASS_NAMES)):
    misclassified = (y_true == true_idx) & (y_pred != true_idx)
    n_errors = misclassified.sum()

    if n_errors == 0:
        ax.text(0.5, 0.5, "No errors", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"True: {true_name}")
        continue

    confused_as = y_pred[misclassified]
    counts      = np.bincount(confused_as, minlength=4).astype(float)
    counts[true_idx] = 0           # zero out self (shouldn't appear)
    fractions   = counts / n_errors

    bars = ax.bar(CLASS_NAMES, fractions, color=COLORS, alpha=0.85)
    bars[true_idx].set_alpha(0.1)  # dim self class visually

    # Annotate bar tops
    for bar, frac in zip(bars, fractions):
        if frac > 0.01:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01,
                f"{frac:.2f}",
                ha="center", va="bottom", fontsize=8,
            )

    ax.set_title(f"True: {true_name}\n({n_errors:,} errors)")
    ax.set_xlabel("Predicted as")
    ax.set_ylabel("Fraction of errors")
    ax.set_ylim(0, 1.15)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

plt.suptitle("Failure analysis: where misclassified nodes go", y=1.02)
plt.tight_layout()
plt.savefig(figures_path / "failure_analysis.png", bbox_inches="tight")
plt.show()